# 2. Lesson 2 Hands-On Lab — Pipeline Orchestration with SageMaker Experiments

This notebook is intentionally simple and executable.

It uses AWS services through `boto3` only — no Apache Airflow, no Step Functions, no simulation classes.

You will build a real ML pipeline orchestrated by **SageMaker Experiments**, covering:
- Creating a **SageMaker Experiment** (the pipeline definition — equivalent to an Airflow DAG)
- Creating a **Trial** (one execution of the pipeline — equivalent to a DAG Run)
- Running each stage as a **Trial Component** (equivalent to a Task) with real AWS status tracking
- Branching on model accuracy using a real gate condition
- Viewing the complete **execution history** in SageMaker Studio's Experiments panel
- Storing all artifacts in **S3** with full lineage tracked in SageMaker
- Publishing pipeline metrics to **CloudWatch**


## 2.1 Environment Setup

### 2.1.1 Import Libraries

In [1]:
# Block 1 - Import libraries

import json
import time
import hashlib
import datetime as dt
from pathlib import Path

import boto3
import joblib
import numpy as np
import pandas as pd

from botocore.exceptions import ClientError
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("Libraries imported successfully")

Libraries imported successfully


### 2.1.2 Connect to AWS

In [2]:
# Block 2 - Create AWS clients

session = boto3.Session()
region  = session.region_name or "us-east-1"

s3               = session.client("s3",         region_name=region)
sts              = session.client("sts",         region_name=region)
cloudwatch       = session.client("cloudwatch",  region_name=region)
sagemaker_client = session.client("sagemaker",   region_name=region)

# sagemaker_client uses boto3 only — not the SageMaker Python SDK

identity   = sts.get_caller_identity()
account_id = identity["Account"]

print("Connected to AWS")
print("Region  :", region)
print("Account :", account_id)
print("Caller  :", identity["Arn"])

Connected to AWS
Region  : eu-north-1
Account : 797715838180
Caller  : arn:aws:sts::797715838180:assumed-role/AmazonSageMakerAdminIAMExecutionRole/SageMaker


### 2.1.3 Configure Project Paths

In [3]:
# Block 3 - Configure project, S3 paths, and experiment names

project_name     = "lesson2-smex-pipeline"
run_id           = dt.datetime.utcnow().strftime("%Y%m%d-%H%M%S")
model_group_name = f"lesson2-loan-risk-{account_id}"

# SageMaker Experiment name = the pipeline definition (one per course/project)
# SageMaker Trial name      = one execution / pipeline run
experiment_name  = f"lesson2-etl-training-pipeline"
trial_name       = f"pipeline-run-{run_id}"

bucket_name = f"{project_name}-{account_id}-{region}".replace("_", "-").lower()

prefix    = "lesson2/smex-pipeline"
raw_key   = f"{prefix}/data/raw/loan_data.csv"
train_key = f"{prefix}/data/processed/train.csv"
test_key  = f"{prefix}/data/processed/test.csv"
model_key = f"{prefix}/model/model_{run_id}.joblib"
run_key   = f"{prefix}/runs/run_{run_id}.json"

local_dir = Path("lesson2_outputs")
local_dir.mkdir(exist_ok=True)

print("Bucket         :", bucket_name)
print("Run ID         :", run_id)
print("Experiment     :", experiment_name)
print("Trial (run)    :", trial_name)
print("Model Registry :", model_group_name)

Bucket         : lesson2-smex-pipeline-797715838180-eu-north-1
Run ID         : 20260628-115709
Experiment     : lesson2-etl-training-pipeline
Trial (run)    : pipeline-run-20260628-115709
Model Registry : lesson2-loan-risk-797715838180


## 2.2 S3 Data Layer

### 2.2.1 Create or Reuse S3 Bucket

In [4]:
# Block 4 - Create or reuse S3 bucket

def bucket_exists(name):
    try:
        s3.head_bucket(Bucket=name)
        return True
    except ClientError as e:
        code = e.response["Error"]["Code"]
        if code in ["404", "NoSuchBucket"]:
            return False
        raise

if not bucket_exists(bucket_name):
    if region == "us-east-1":
        s3.create_bucket(Bucket=bucket_name)
    else:
        s3.create_bucket(
            Bucket=bucket_name,
            CreateBucketConfiguration={"LocationConstraint": region}
        )
    print("Created bucket:", bucket_name)
else:
    print("Using existing bucket:", bucket_name)

s3.put_bucket_versioning(
    Bucket=bucket_name,
    VersioningConfiguration={"Status": "Enabled"}
)
print("S3 versioning enabled")

Created bucket: lesson2-smex-pipeline-797715838180-eu-north-1
S3 versioning enabled


### 2.2.2 Generate Loan Risk Dataset and Upload to S3

In [5]:
# Block 5 - Generate and upload raw dataset to S3

columns = [
    "income_score",
    "credit_history_score",
    "debt_ratio_score",
    "employment_score",
    "savings_score",
    "repayment_behavior_score"
]

X, y = make_classification(
    n_samples=1500,
    n_features=6,
    n_informative=4,
    n_redundant=1,
    n_classes=2,
    random_state=42
)

df = pd.DataFrame(X, columns=columns)
df["loan_default_risk"] = y

raw_path = local_dir / "loan_data.csv"
df.to_csv(raw_path, index=False)
s3.upload_file(str(raw_path), bucket_name, raw_key)

display(df.head())
print("Rows    :", len(df))
print("Columns :", list(df.columns))
print("Uploaded:", f"s3://{bucket_name}/{raw_key}")

,income_score,credit_history_score,debt_ratio_score,employment_score,savings_score,repayment_behavior_score,loan_default_risk
0,1.705099,1.004242,0.588320,-0.410098,-0.282672,-2.320010,1
1,0.918796,-0.262210,1.302603,2.873578,-3.560945,-0.354622,0
2,0.526696,0.524000,-1.071553,0.197359,0.052271,-1.531038,0
3,-0.719885,0.205568,-1.516909,-0.078276,0.812511,0.107889,1
4,1.487276,-1.031392,0.447000,0.456336,-1.478771,-2.083008,1


Rows    : 1500
Columns : ['income_score', 'credit_history_score', 'debt_ratio_score', 'employment_score', 'savings_score', 'repayment_behavior_score', 'loan_default_risk']
Uploaded: s3://lesson2-smex-pipeline-797715838180-eu-north-1/lesson2/smex-pipeline/data/raw/loan_data.csv


## 2.3 SageMaker Experiment Setup

### What is a SageMaker Experiment?

SageMaker Experiments maps directly onto pipeline orchestration concepts:

| Orchestration Concept | SageMaker Experiments Equivalent |
|---|---|
| DAG (pipeline definition) | **Experiment** — created once, shared across all runs |
| DAG Run (one execution) | **Trial** — one trial per pipeline execution |
| Task (pipeline stage) | **Trial Component** — one per stage, with real AWS status |
| XCom / context passing | **Component Parameters & Artifacts** — tracked in AWS |
| Execution history | **List Trial Components** API — queryable from anywhere |

Every stage (extract → validate → transform → train → evaluate → register) creates a real AWS Trial Component with an ARN, status, and full metadata stored in SageMaker.

### 2.3.1 Create the Experiment (Pipeline Definition)


In [6]:
# Block 6 - Create SageMaker Experiment — the pipeline definition

try:
    sagemaker_client.create_experiment(
        ExperimentName=experiment_name,
        Description=(
            "FinSight AI — Loan default risk ETL-to-Training pipeline. "
            "Tracks all pipeline runs (trials) and their stages (trial components)."
        ),
        Tags=[{"Key": "Project", "Value": project_name}]
    )
    print("Created experiment:", experiment_name)
except ClientError as e:
    if e.response["Error"]["Code"] == "ConflictException":
        print("Experiment already exists — reusing:", experiment_name)
    else:
        raise

print()
print("View in SageMaker Studio: Experiments panel →", experiment_name)

Created experiment: lesson2-etl-training-pipeline

View in SageMaker Studio: Experiments panel → lesson2-etl-training-pipeline


### 2.3.2 Create the Trial (This Pipeline Run)

In [7]:
# Block 7 - Create SageMaker Trial — one execution of the pipeline

sagemaker_client.create_trial(
    ExperimentName=experiment_name,
    TrialName=trial_name,
    Tags=[
        {"Key": "RunId",   "Value": run_id},
        {"Key": "Project", "Value": project_name}
    ]
)

print("Created trial:", trial_name)
print("Experiment   :", experiment_name)
print()
print("This trial will collect all pipeline stages for run:", run_id)

Created trial: pipeline-run-20260628-115709
Experiment   : lesson2-etl-training-pipeline

This trial will collect all pipeline stages for run: 20260628-115709


## 2.4 Pipeline Stage Runner

The helper function below handles every pipeline stage in a consistent way:
1. **Creates a Trial Component** in SageMaker (real AWS resource, real ARN)
2. **Associates it** with the current Trial (linking this stage to this run)
3. **Sets status to InProgress** — visible immediately in SageMaker Studio
4. **Runs the ML logic** locally
5. **Updates status to Completed or Failed** — recorded permanently in AWS
6. **Stores input/output artifact S3 paths** and numeric parameters in the component


In [8]:
# Block 8 - Define pipeline stage runner

def run_stage(stage_id, display_name, worker_fn, context,
              input_artifacts=None, param_keys=None):
    """
    Execute one pipeline stage tracked as a SageMaker Trial Component.

    Parameters
    ----------
    stage_id        : str   Short unique ID, e.g. "extract-data"
    display_name    : str   Human-readable name shown in SageMaker Studio
    worker_fn       : func  Function that does the actual ML work
    context         : dict  Shared state between stages (equivalent to XComs)
    input_artifacts : dict  S3 URIs to register as inputs {name: s3_uri}
    param_keys      : list  Keys from context to log as numeric parameters
    """
    component_name = f"{stage_id}-{run_id}"
    now            = dt.datetime.utcnow()

    # ── Step 1: Create Trial Component in SageMaker ──────────────────────────
    sagemaker_client.create_trial_component(
        TrialComponentName=component_name,
        DisplayName=display_name,
        Status={"PrimaryStatus": "InProgress"},
        StartTime=now,
        InputArtifacts={
            name: {"Value": uri, "MediaType": "text/plain"}
            for name, uri in (input_artifacts or {}).items()
        }
    )

    # ── Step 2: Associate with current trial (pipeline run) ──────────────────
    sagemaker_client.associate_trial_component(
        TrialName=trial_name,
        TrialComponentName=component_name
    )

    print(f"  ▶  [{display_name}] started — tracking in SageMaker")

    try:
        # ── Step 3: Execute the ML logic ─────────────────────────────────────
        result = worker_fn(context) or {}

        # ── Step 4: Collect output S3 paths and numeric parameters ───────────
        out_artifacts = {
            k: {"Value": v, "MediaType": "text/plain"}
            for k, v in result.items()
            if isinstance(v, str) and v.startswith("s3://")
        }
        params = {}
        for k in (param_keys or []):
            val = context.get(k)
            if isinstance(val, (int, float)):
                params[k] = {"NumberValue": float(val)}
            elif val is not None:
                params[k] = {"StringValue": str(val)}

        # ── Step 5: Mark Completed with outputs ──────────────────────────────
        sagemaker_client.update_trial_component(
            TrialComponentName=component_name,
            Status={"PrimaryStatus": "Completed"},
            EndTime=dt.datetime.utcnow(),
            OutputArtifacts=out_artifacts,
            Parameters=params
        )
        print(f"  ✅ [{display_name}] completed — status updated in SageMaker")
        return result

    except Exception as exc:
        # ── Step 6: Mark Failed on error ─────────────────────────────────────
        sagemaker_client.update_trial_component(
            TrialComponentName=component_name,
            Status={"PrimaryStatus": "Failed"},
            EndTime=dt.datetime.utcnow()
        )
        print(f"  ❌ [{display_name}] failed: {exc}")
        raise

print("run_stage helper defined")

run_stage helper defined


## 2.5 Define Worker Functions

Each worker does the ML logic for its stage and returns a dict of outputs.

In [9]:
# Block 9 - Define all pipeline worker functions

# ── Worker 1: Extract Data ─────────────────────────────────────────────────

def worker_extract(ctx):
    df = pd.read_csv(local_dir / "loan_data.csv")
    ctx["row_count"]   = int(len(df))
    ctx["raw_s3_path"] = f"s3://{bucket_name}/{raw_key}"
    print(f"    {len(df)} rows available")
    return {"raw_data": ctx["raw_s3_path"]}


# ── Worker 2: Validate Data ────────────────────────────────────────────────

def worker_validate(ctx):
    df = pd.read_csv(local_dir / "loan_data.csv")
    required = columns + ["loan_default_risk"]

    missing = [c for c in required if c not in df.columns]
    assert not missing,                      f"Missing columns: {missing}"
    assert df.isna().sum().sum() == 0,       "Dataset contains nulls"
    assert df["loan_default_risk"].isin([0, 1]).all(), "Target must be 0 or 1"
    assert len(df) >= 500,                   f"Too few rows: {len(df)}"

    dist = df["loan_default_risk"].value_counts().to_dict()
    ctx["null_count"] = 0
    print(f"    All checks passed — rows={len(df)}, class dist={dist}")
    return {}


# ── Worker 3: Transform Data ───────────────────────────────────────────────

def worker_transform(ctx):
    df        = pd.read_csv(local_dir / "loan_data.csv")
    feat_cols = [c for c in df.columns if c != "loan_default_risk"]
    X, y      = df[feat_cols], df["loan_default_risk"]

    scaler    = StandardScaler()
    X_scaled  = pd.DataFrame(scaler.fit_transform(X), columns=feat_cols)

    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.25, random_state=42, stratify=y
    )

    train_df = X_train.copy(); train_df["loan_default_risk"] = y_train.values
    test_df  = X_test.copy();  test_df["loan_default_risk"]  = y_test.values

    train_path, test_path = local_dir / "train.csv", local_dir / "test.csv"
    train_df.to_csv(train_path, index=False)
    test_df.to_csv(test_path, index=False)

    s3.upload_file(str(train_path), bucket_name, train_key)
    s3.upload_file(str(test_path),  bucket_name, test_key)

    joblib.dump(scaler, local_dir / "scaler.joblib")
    ctx["feat_cols"]  = feat_cols
    ctx["train_rows"] = int(len(train_df))
    ctx["test_rows"]  = int(len(test_df))

    print(f"    train={len(train_df)}, test={len(test_df)}, features={len(feat_cols)}")
    return {
        "train_data": f"s3://{bucket_name}/{train_key}",
        "test_data":  f"s3://{bucket_name}/{test_key}"
    }


# ── Worker 4: Train Model ──────────────────────────────────────────────────

def worker_train(ctx):
    train_df  = pd.read_csv(local_dir / "train.csv")
    feat_cols = [c for c in train_df.columns if c != "loan_default_risk"]

    model = RandomForestClassifier(
        n_estimators=100, max_depth=6, random_state=42, class_weight="balanced"
    )
    model.fit(train_df[feat_cols], train_df["loan_default_risk"])

    model_path = local_dir / "model.joblib"
    joblib.dump(model, model_path)

    with open(model_path, "rb") as f:
        model_hash = hashlib.sha256(f.read()).hexdigest()

    s3.upload_file(str(model_path), bucket_name, model_key)
    ctx["model_hash"] = model_hash

    print(f"    Model trained — SHA256: {model_hash[:16]}...")
    return {"model_artifact": f"s3://{bucket_name}/{model_key}"}


# ── Worker 5: Evaluate Model ───────────────────────────────────────────────

def worker_evaluate(ctx):
    test_df   = pd.read_csv(local_dir / "test.csv")
    feat_cols = [c for c in test_df.columns if c != "loan_default_risk"]

    model    = joblib.load(local_dir / "model.joblib")
    preds    = model.predict(test_df[feat_cols])
    accuracy = float(accuracy_score(test_df["loan_default_risk"], preds))

    print(classification_report(test_df["loan_default_risk"], preds))
    print(f"    Accuracy: {accuracy:.4f}")

    ctx["accuracy"] = accuracy
    return {}


# ── Worker 6: Register Model ───────────────────────────────────────────────

def worker_register(ctx):
    accuracy = float(ctx.get("accuracy", 0))

    # Create model package group — safe to re-run
    try:
        sagemaker_client.create_model_package_group(
            ModelPackageGroupName=model_group_name,
            ModelPackageGroupDescription="Lesson 2 — Loan default risk models"
        )
    except ClientError as e:
        if e.response["Error"]["Code"] == "ConflictException":
            pass
        else:
            raise

    reg_response = sagemaker_client.create_model_package(
        ModelPackageGroupName=model_group_name,
        ModelPackageDescription=f"RFC | accuracy={accuracy:.4f} | run={run_id}",
        ModelApprovalStatus="Approved",
        CustomerMetadataProperties={
            "accuracy":       str(round(accuracy, 4)),
            "model_artifact": f"s3://{bucket_name}/{model_key}",
            "model_hash":     ctx.get("model_hash", "")[:32],
            "train_data":     f"s3://{bucket_name}/{train_key}",
            "test_data":      f"s3://{bucket_name}/{test_key}",
            "run_id":         run_id,
            "experiment":     experiment_name,
            "trial":          trial_name
        }
    )

    pkg_arn = reg_response["ModelPackageArn"]
    ctx["model_package_arn"] = pkg_arn
    print(f"    Registered in SageMaker Model Registry")
    print(f"    ModelPackageArn: {pkg_arn}")
    return {"model_package_arn": pkg_arn}


# ── Worker 7: Flag for Retraining ─────────────────────────────────────────

def worker_flag_retrain(ctx):
    accuracy = float(ctx.get("accuracy", 0))
    alert = {
        "alert_type": "accuracy_gate_failed",
        "accuracy":   accuracy,
        "threshold":  0.75,
        "action":     "Retrain with updated data or tuned hyperparameters",
        "run_id":     run_id,
        "timestamp":  dt.datetime.utcnow().isoformat()
    }
    alert_path = local_dir / "retrain_alert.json"
    alert_path.write_text(json.dumps(alert, indent=2))
    s3.upload_file(str(alert_path), bucket_name,
                   f"{prefix}/alerts/retrain_alert_{run_id}.json")
    print(f"    Alert raised — accuracy {accuracy:.4f} < 0.75")
    return {}


print("All worker functions defined")

All worker functions defined


## 2.6 Execute the Pipeline

Each stage call:
1. Creates a Trial Component in SageMaker (real AWS resource with ARN)
2. Runs the ML logic
3. Records status + artifact paths + parameters back to SageMaker


In [11]:
# Block 10 - Run the full pipeline

context = {}   # shared state between stages

print(f"Pipeline run  : {run_id}")
print(f"Experiment    : {experiment_name}")
print(f"Trial         : {trial_name}")
print(f"{'='*60}")
print()

# ── Stage 1: Extract ──────────────────────────────────────────────────────
run_stage(
    "extract-data", "Stage-1-Extract-Data",
    worker_extract, context,
    param_keys=["row_count"]
)

# ── Stage 2: Validate ─────────────────────────────────────────────────────
run_stage(
    "validate-data", "Stage-2-Validate-Data",
    worker_validate, context,
    input_artifacts={"raw-data": context.get("raw_s3_path", "")},
    param_keys=["row_count", "null_count"]
)

# ── Stage 3: Transform ────────────────────────────────────────────────────
run_stage(
    "transform-data", "Stage-3-Transform-Data",
    worker_transform, context,
    input_artifacts={"raw-data": context.get("raw_s3_path", "")},
    param_keys=["train_rows", "test_rows"]
)

# ── Stage 4: Train ────────────────────────────────────────────────────────
run_stage(
    "train-model", "Stage-4-Train-Model",
    worker_train, context,
    input_artifacts={
        "train-data": f"s3://{bucket_name}/{train_key}",
        "test-data":  f"s3://{bucket_name}/{test_key}"
    },
    param_keys=["train_rows"]
)

# ── Stage 5: Evaluate ─────────────────────────────────────────────────────
run_stage(
    "evaluate-model", "Stage-5-Evaluate-Model",
    worker_evaluate, context,
    input_artifacts={"model-artifact": f"s3://{bucket_name}/{model_key}"},
    param_keys=["accuracy"]
)

# ── Stage 6: Branch on accuracy gate ──────────────────────────────────────
print()
accuracy = context.get("accuracy", 0)
GATE     = 0.75

if accuracy >= GATE:
    print(f"  Accuracy {accuracy:.4f} >= {GATE} — running RegisterModel branch")
    run_stage(
        "register-model", "Stage-6-Register-Model",
        worker_register, context,
        input_artifacts={"model-artifact": f"s3://{bucket_name}/{model_key}"},
        param_keys=["accuracy"]
    )
else:
    print(f"  Accuracy {accuracy:.4f} < {GATE} — running FlagRetraining branch")
    run_stage(
        "flag-retraining", "Stage-6-Flag-Retraining",
        worker_flag_retrain, context,
        param_keys=["accuracy"]
    )

print()
print(f"{'='*60}")
print("Pipeline complete")

Pipeline run  : 20260628-115709
Experiment    : lesson2-etl-training-pipeline
Trial         : pipeline-run-20260628-115709

  ▶  [Stage-1-Extract-Data] started — tracking in SageMaker
    1500 rows available
  ✅ [Stage-1-Extract-Data] completed — status updated in SageMaker
  ▶  [Stage-2-Validate-Data] started — tracking in SageMaker
    All checks passed — rows=1500, class dist={0: 751, 1: 749}
  ✅ [Stage-2-Validate-Data] completed — status updated in SageMaker
  ▶  [Stage-3-Transform-Data] started — tracking in SageMaker
    train=1125, test=375, features=6
  ✅ [Stage-3-Transform-Data] completed — status updated in SageMaker
  ▶  [Stage-4-Train-Model] started — tracking in SageMaker
    Model trained — SHA256: 88a9364e668e90b1...
  ✅ [Stage-4-Train-Model] completed — status updated in SageMaker
  ▶  [Stage-5-Evaluate-Model] started — tracking in SageMaker
              precision    recall  f1-score   support

           0       0.86      0.86      0.86       188
           1       0.

## 2.7 Query the Pipeline Execution from SageMaker

### 2.7.1 List All Trial Components (Audit Trail)

In [12]:
# Block 11 - Query all trial components from SageMaker

response = sagemaker_client.list_trial_components(
    TrialName=trial_name,
    SortBy="CreationTime",
    SortOrder="Ascending"
)

components = response["TrialComponentSummaries"]

print(f"SageMaker Trial Components for run: {trial_name}")
print(f"{'─'*70}")
print(f"  {'Stage':<35} {'Status':<12} {'Created'}")
print(f"{'─'*70}")

for comp in components:
    name    = comp.get("DisplayName") or comp["TrialComponentName"]
    status  = comp.get("Status", {}).get("PrimaryStatus", "Unknown")
    created = comp["CreationTime"].strftime("%H:%M:%S")
    icon    = "✅" if status == "Completed" else ("❌" if status == "Failed" else "⏳")
    print(f"  {icon}  {name:<33} {status:<12} {created}")

print(f"{'─'*70}")
print(f"Total stages: {len(components)}")
print()
print("View in SageMaker Studio → Experiments →", experiment_name)

SageMaker Trial Components for run: pipeline-run-20260628-115709
──────────────────────────────────────────────────────────────────────
  Stage                               Status       Created
──────────────────────────────────────────────────────────────────────
  ✅  Stage-1-Extract-Data              Completed    11:59:58
  ✅  Stage-2-Validate-Data             Completed    11:59:59
  ✅  Stage-3-Transform-Data            Completed    12:00:00
  ✅  Stage-4-Train-Model               Completed    12:00:03
  ✅  Stage-5-Evaluate-Model            Completed    12:00:05
  ✅  Stage-6-Register-Model            Completed    12:00:06
──────────────────────────────────────────────────────────────────────
Total stages: 6

View in SageMaker Studio → Experiments → lesson2-etl-training-pipeline


### 2.7.2 Inspect a Specific Stage Component

In [13]:
# Block 12 - Inspect the Train Model component in detail

component_name = f"train-model-{run_id}"

detail = sagemaker_client.describe_trial_component(
    TrialComponentName=component_name
)

print("Trial Component:", detail["DisplayName"])
print("Status         :", detail["Status"]["PrimaryStatus"])
print("Start          :", detail["StartTime"].strftime("%Y-%m-%d %H:%M:%S UTC"))
print("End            :", detail.get("EndTime", dt.datetime.utcnow()).strftime("%Y-%m-%d %H:%M:%S UTC"))
print()

if detail.get("InputArtifacts"):
    print("Input Artifacts:")
    for k, v in detail["InputArtifacts"].items():
        print(f"  {k}: {v['Value']}")

if detail.get("OutputArtifacts"):
    print("Output Artifacts:")
    for k, v in detail["OutputArtifacts"].items():
        print(f"  {k}: {v['Value']}")

if detail.get("Parameters"):
    print("Parameters:")
    for k, v in detail["Parameters"].items():
        val = v.get("NumberValue", v.get("StringValue", ""))
        print(f"  {k}: {val}")

Trial Component: Stage-4-Train-Model
Status         : Completed
Start          : 2026-06-28 12:00:03 UTC
End            : 2026-06-28 12:00:05 UTC

Input Artifacts:
  test-data: s3://lesson2-smex-pipeline-797715838180-eu-north-1/lesson2/smex-pipeline/data/processed/test.csv
  train-data: s3://lesson2-smex-pipeline-797715838180-eu-north-1/lesson2/smex-pipeline/data/processed/train.csv
Output Artifacts:
  model_artifact: s3://lesson2-smex-pipeline-797715838180-eu-north-1/lesson2/smex-pipeline/model/model_20260628-115709.joblib
Parameters:
  train_rows: 1125.0


## 2.8 Monitoring

### 2.8.1 Publish Pipeline Metrics to CloudWatch

In [14]:
# Block 13 - Publish pipeline run metrics to CloudWatch

namespace = "Lesson2/SMExPipeline"

cloudwatch.put_metric_data(
    Namespace=namespace,
    MetricData=[
        {
            "MetricName": "PipelineStagesCompleted",
            "Value":      float(sum(
                1 for c in components
                if c.get("Status", {}).get("PrimaryStatus") == "Completed"
            )),
            "Unit":       "Count",
            "Dimensions": [{"Name": "Project", "Value": project_name}]
        },
        {
            "MetricName": "ModelAccuracy",
            "Value":      float(context.get("accuracy", 0)),
            "Unit":       "None",
            "Dimensions": [{"Name": "RunId", "Value": run_id}]
        },
        {
            "MetricName": "TrainingRows",
            "Value":      float(context.get("train_rows", 0)),
            "Unit":       "Count",
            "Dimensions": [{"Name": "Project", "Value": project_name}]
        }
    ]
)

print("CloudWatch metrics published")
print("Namespace:", namespace)
print(f"  PipelineStagesCompleted : {sum(1 for c in components if c.get('Status', {}).get('PrimaryStatus') == 'Completed')}")
print(f"  ModelAccuracy           : {context.get('accuracy', 0):.4f}")
print(f"  TrainingRows            : {context.get('train_rows', 0)}")

CloudWatch metrics published
Namespace: Lesson2/SMExPipeline
  PipelineStagesCompleted : 6
  ModelAccuracy           : 0.8560
  TrainingRows            : 1125


### 2.8.2 List All S3 Artifacts

In [15]:
# Block 14 - List all pipeline artifacts in S3

print("S3 artifacts created by this pipeline run:")
print()

response = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)
objects  = response.get("Contents", [])

for obj in objects:
    print(f"  {obj['Key']}")
    print(f"    {obj['Size'] / 1024:.1f} KB  |  {obj['LastModified'].strftime('%Y-%m-%d %H:%M:%S')} UTC")

print()
print("Total artifacts:", len(objects))

S3 artifacts created by this pipeline run:

  lesson2/smex-pipeline/data/processed/test.csv
    43.9 KB  |  2026-06-28 12:00:04 UTC
  lesson2/smex-pipeline/data/processed/train.csv
    131.6 KB  |  2026-06-28 12:00:04 UTC
  lesson2/smex-pipeline/data/raw/loan_data.csv
    174.5 KB  |  2026-06-28 11:57:16 UTC
  lesson2/smex-pipeline/model/model_20260628-115709.joblib
    650.1 KB  |  2026-06-28 12:00:06 UTC

Total artifacts: 4


## 2.9 Pipeline Run Summary

In [16]:
# Block 15 - Write and upload pipeline run summary

run_summary = {
    "pipeline":         "lesson2-smex-etl-training",
    "run_id":           run_id,
    "aws_region":       region,
    "aws_account":      account_id,
    "sagemaker_experiment": experiment_name,
    "sagemaker_trial":  trial_name,
    "stages_total":     len(components),
    "stages_completed": sum(1 for c in components if c.get("Status", {}).get("PrimaryStatus") == "Completed"),
    "data": {
        "raw_s3":     f"s3://{bucket_name}/{raw_key}",
        "train_s3":   f"s3://{bucket_name}/{train_key}",
        "test_s3":    f"s3://{bucket_name}/{test_key}",
        "row_count":  context.get("row_count"),
        "train_rows": context.get("train_rows"),
        "test_rows":  context.get("test_rows"),
    },
    "model": {
        "artifact":         f"s3://{bucket_name}/{model_key}",
        "sha256":           context.get("model_hash"),
        "accuracy":         context.get("accuracy"),
        "model_package_arn": context.get("model_package_arn")
    },
    "completed_at": dt.datetime.utcnow().isoformat()
}

run_path = local_dir / f"run_summary_{run_id}.json"
run_path.write_text(json.dumps(run_summary, indent=2))
s3.upload_file(str(run_path), bucket_name, run_key)

print(json.dumps(run_summary, indent=2))
print()
print("Run summary uploaded:", f"s3://{bucket_name}/{run_key}")

{
  "pipeline": "lesson2-smex-etl-training",
  "run_id": "20260628-115709",
  "aws_region": "eu-north-1",
  "aws_account": "797715838180",
  "sagemaker_experiment": "lesson2-etl-training-pipeline",
  "sagemaker_trial": "pipeline-run-20260628-115709",
  "stages_total": 6,
  "stages_completed": 6,
  "data": {
    "raw_s3": "s3://lesson2-smex-pipeline-797715838180-eu-north-1/lesson2/smex-pipeline/data/raw/loan_data.csv",
    "train_s3": "s3://lesson2-smex-pipeline-797715838180-eu-north-1/lesson2/smex-pipeline/data/processed/train.csv",
    "test_s3": "s3://lesson2-smex-pipeline-797715838180-eu-north-1/lesson2/smex-pipeline/data/processed/test.csv",
    "row_count": 1500,
    "train_rows": 1125,
    "test_rows": 375
  },
  "model": {
    "artifact": "s3://lesson2-smex-pipeline-797715838180-eu-north-1/lesson2/smex-pipeline/model/model_20260628-115709.joblib",
    "sha256": "88a9364e668e90b10aeff0335bc161fe1fb6abcf4de67df4efe763530009affc",
    "accuracy": 0.856,
    "model_package_arn": "ar

## 2.10 Cleanup

### 2.10.1 Optional Cleanup

In [17]:
# Block 16 - Optional cleanup
# Set CLEANUP = True to delete all resources created by this lab.

CLEANUP = False

if CLEANUP:
    # Delete S3 objects
    objs = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix).get("Contents", [])
    if objs:
        s3.delete_objects(
            Bucket=bucket_name,
            Delete={"Objects": [{"Key": o["Key"]} for o in objs]}
        )
        print(f"Deleted {len(objs)} S3 objects")

    # Delete trial components
    for comp in sagemaker_client.list_trial_components(
        TrialName=trial_name
    ).get("TrialComponentSummaries", []):
        sagemaker_client.disassociate_trial_component(
            TrialName=trial_name,
            TrialComponentName=comp["TrialComponentName"]
        )
        sagemaker_client.delete_trial_component(
            TrialComponentName=comp["TrialComponentName"]
        )
        print("Deleted component:", comp["TrialComponentName"])

    # Delete trial and experiment
    sagemaker_client.delete_trial(
        ExperimentName=experiment_name, TrialName=trial_name
    )
    print("Deleted trial:", trial_name)

    sagemaker_client.delete_experiment(ExperimentName=experiment_name)
    print("Deleted experiment:", experiment_name)

    # SageMaker Model Registry cleanup
    pkgs = sagemaker_client.list_model_packages(
        ModelPackageGroupName=model_group_name, MaxResults=100
    ).get("ModelPackageSummaryList", [])
    for pkg in pkgs:
        sagemaker_client.delete_model_package(
            ModelPackageName=pkg["ModelPackageArn"]
        )
    if pkgs:
        sagemaker_client.delete_model_package_group(
            ModelPackageGroupName=model_group_name
        )
        print("Deleted model package group:", model_group_name)

else:
    print("Cleanup skipped")
    print("Set CLEANUP = True and rerun to delete all lab resources")

Cleanup skipped
Set CLEANUP = True and rerun to delete all lab resources


### 2.10.2 Final Checklist

This notebook covered — using **real AWS services only**:

- **AWS S3** — bucket, raw data, train/test CSVs, model artifact, alerts, run summary
- **SageMaker Experiments** — experiment created (pipeline definition), trial created (pipeline run), each stage created as a Trial Component with real ARN, real status (InProgress → Completed/Failed), input/output artifact S3 URIs, and numeric parameters all stored in AWS
- **SageMaker Model Registry** — model version registered with `ModelApprovalStatus`, accuracy, and full metadata via `create_model_package`
- **CloudWatch** — pipeline metrics published (stages completed, accuracy, training rows)

No Step Functions. No custom simulation classes. All pipeline orchestration state stored and queryable from AWS SageMaker.
